# Counterexample Commons - Complete Colab Lab

Safe no-key launcher for the local-first Gradio research lab.

[Open in GitHub](https://github.com/error-wtf/counterexample-commons/tree/rescue/integrated-complete-lab)


## Scope and safety

This notebook launches the same Gradio app from the rescue branch in `colab-public-demo` mode.

It is designed for a fresh Google Colab runtime with no provider credentials loaded.

Scientific boundary:

- Exact finite baseline calculations are locally executable.
- The finite rational mesh baseline is not Sawin's construction.
- Finite exact validation is not evidence for exponent n^1.014.
- OpenAI/Sawin asymptotic results are source-documented only here, not locally reproduced.
- AI/provider live workflows are not part of this public Colab demo.


## Sources and theorem map

| Result | Source | Local status |
|---|---|---|
| Erdős unit-distance problem | Historical/source context | Source documented |
| OpenAI fixed-delta disproof | OpenAI announcement and proof PDF | SOURCE_DOCUMENTED; not locally reproduced |
| Original OpenAI proof explicit delta = 0.014 | Original proof does not provide this value | NOT_PROVIDED_BY_ORIGINAL_PROOF |
| Sawin explicit exponent n^1.014 | Will Sawin, arXiv:2605.20579 | SOURCE_DOCUMENTED; not locally reproduced |
| Finite rational mesh baseline | Repository exact validator | LOCALLY_REPRODUCED_EXACT finite baseline only |


## 1. Fresh runtime setup

Run this cell first. In Colab it clones the rescue branch and installs the package. Locally it only locates the current checkout for precheck use.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = (
    "google.colab" in sys.modules
    or bool(os.environ.get("COLAB_RELEASE_TAG"))
    or bool(os.environ.get("COLAB_BACKEND_VERSION"))
)

REPO_URL = "https://github.com/error-wtf/counterexample-commons.git"
REPO_BRANCH = "rescue/integrated-complete-lab"
COLAB_REPO_DIR = Path("/content/counterexample-commons")
PROVIDER_ENV_VARS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "OPENROUTER_API_KEY",
    "GEMINI_API_KEY",
    "MISTRAL_API_KEY",
    "OLLAMA_API_KEY",
]

loaded_provider_vars = [
    name for name in PROVIDER_ENV_VARS
    if os.environ.get(name)
]
if loaded_provider_vars:
    raise RuntimeError(
        "Provider credential environment variables are present. "
        "Do not launch the public Colab demo from a credentialed runtime. "
        "Start a fresh runtime or remove credentials first."
    )

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", REPO_BRANCH, REPO_URL,
                str(COLAB_REPO_DIR),
            ],
            check=True,
        )
    elif (COLAB_REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            [
                "git", "-C", str(COLAB_REPO_DIR), "checkout", "-B",
                REPO_BRANCH, f"origin/{REPO_BRANCH}",
            ],
            check=True,
        )
    else:
        raise RuntimeError(
            f"{COLAB_REPO_DIR} exists but is not a Git checkout. "
            "Move it aside or restart the runtime."
        )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(COLAB_REPO_DIR)],
        check=True,
    )
    repo_root = COLAB_REPO_DIR
    runtime_label = "GOOGLE COLAB PUBLIC NO-KEY DEMO"
else:
    start = Path.cwd().resolve()
    repo_root = next(
        (
            p for p in [start, *start.parents]
            if (p / "counterexample_commons").is_dir()
            and (p / "pyproject.toml").is_file()
        ),
        None,
    )
    if repo_root is None:
        raise RuntimeError("Repository root not found for local precheck.")
    runtime_label = "LOCAL PRECHECK ONLY - NOT GOOGLE COLAB EVIDENCE"

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import counterexample_commons

print(runtime_label)
print(f"Repository branch: {REPO_BRANCH}")
print(f"Package version: {counterexample_commons.__version__}")
print("Provider credentials: not configured for this public demo")


## 2. Exact finite self-test

This cell verifies the exact rational validator and the finite rational mesh baseline before the UI is launched.


In [ ]:
from sympy import Rational
from counterexample_commons.validators import (
    count_unit_edges_exact,
    validate_grid_configuration,
    validate_line_configuration,
    validate_custom_configuration,
)
from case_studies.erdos_unit_distance_2026.rational_mesh_baseline import (
    verify_rational_mesh,
)

pair_points = [
    (Rational(0), Rational(0)),
    (Rational(3, 5), Rational(4, 5)),
]
pair_count, pair_edges = count_unit_edges_exact(pair_points)
assert pair_count == 1
assert pair_edges == [(0, 1)]

line = validate_line_configuration(5)
assert line["pass"] is True

grid = validate_grid_configuration(5)
assert grid["pass"] is True

rational_example = validate_custom_configuration([
    ("0", "0"),
    ("3/5", "4/5"),
])
assert rational_example["actual_edges"] == 1

mesh = verify_rational_mesh(10)
assert mesh["n"] == 121
assert mesh["unit_edges"] == 82

print("Exact validator self-test: PASS")
print("Rational 3/5-4/5 example: 1 exact unit-distance edge")
print("Finite Rational Mesh Baseline m=10: 121 points")
print("Finite Rational Mesh Baseline m=10: 82 exactly validated unit-distance edges")
print("Finite rational mesh baseline - not Sawin's construction.")
print("Finite exact validation only; not evidence for exponent n^1.014.")


## 3. Launch the complete Gradio lab

This starts the app in `colab-public-demo` mode.

The public demo exposes exact finite baselines, the configuration explorer, visualisation, read-only claims, and finite exports. AI/provider live calls are disabled in this mode.


In [ ]:
from app.main import build_app
from counterexample_commons.config import AppMode

print("Launching Counterexample Commons in colab-public-demo mode.")
print("No secrets. No live providers. Claim registry read-only.")
print("AI/provider live workflows are disabled in this public Colab demo.")

app = build_app(AppMode.COLAB_PUBLIC_DEMO)

if IN_COLAB:
    app.queue().launch(share=True, debug=False, show_error=True)
else:
    print("Local precheck: app built successfully.")
    print("In Google Colab this cell launches with share=True.")


## 4. Optional private AI workflow is intentionally not enabled here

This notebook is the safe public no-key entry point. Do not paste provider keys into notebook cells.

Private live-provider testing belongs in a private local `.env` workflow or a separate explicitly private Colab session. AI-generated candidates remain finite hypotheses until independently checked by the exact validator.


## 5. Runtime status

Expected public-demo status:

```text
COLAB_MODE: colab-public-demo
PROVIDER_KEYS: not configured
LIVE_PROVIDER_CALLS: disabled
CLAIM_REGISTRY: read-only
SAWIN_N_1_014: SOURCE_DOCUMENTED; not locally reproduced
FINITE_RATIONAL_MESH: LOCALLY_REPRODUCED_EXACT finite baseline only
```
